# Deep Learning 023 — Normalizing Inputs

Companion notebook to the lesson. Feature scaling is one line of code, it changes no
architecture, and on badly scaled data it is the difference between a model that works and
one that does not.

This is the lesson's dataset: `Age` spans about 42 years, `EstimatedSalary` spans about
135,000 rupees. Same units? No. Same scale? Nowhere near.

| Claim | Measured below |
|---|---|
| the spans differ by three orders of magnitude | **3,214 to 1** |
| unscaled training fails | **0.360** test accuracy against **0.900** scaled — below the 0.640 baseline |
| the loss surface is the reason | condition number **10.8 million** → **1.4** |
| gradient descent on it does not just stall | the raw loss *rises*, 9.82 → 17.72 |
| fit the scaler on train only | the test set's statistics come out imperfect, and should |

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler

df = pd.read_csv("../data/Social_Network_Ads.csv")
X = df[["Age", "EstimatedSalary"]].values.astype(float)
y = df["Purchased"].values
print(df[["Age", "EstimatedSalary", "Purchased"]].describe().T[["min", "max", "mean", "std"]]
      .round(1).to_string())

spans = X.max(0) - X.min(0)
print(f"\nAge spans {spans[0]:,.0f}, EstimatedSalary spans {spans[1]:,.0f}")
print(f"ratio: {spans[1] / spans[0]:,.0f} to 1")

## Part A — What the network actually sees

A neuron computes `w1*Age + w2*Salary + b`. Both weights start from the same small random
distribution, so before any learning happens the salary term is thousands of times louder
than the age term — **not because salary matters more, but because it is measured in bigger
numbers.**

In [ ]:
rng = np.random.default_rng(0)
w = rng.normal(scale=0.05, size=2)                 # a typical initialisation
contrib = np.abs(X * w)
print(f"typical |w_i * x_i| at initialisation:")
print(f"   Age             {contrib[:, 0].mean():>12,.2f}")
print(f"   EstimatedSalary {contrib[:, 1].mean():>12,.2f}")
print(f"   ratio           {contrib[:, 1].mean() / contrib[:, 0].mean():>12,.0f} to 1")
print("\nThe gradient inherits the same imbalance, so the optimiser spends its whole")
print("budget adjusting one weight while the other barely moves.")

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

def run(A, B, label, seed=0):
    m = MLPClassifier(hidden_layer_sizes=(8,), max_iter=300, random_state=seed,
                      learning_rate_init=0.01)
    m.fit(A, y_tr)
    return m.score(B, y_te)

print(f"{'inputs':<22}{'test accuracy':>15}")
print(f"{'raw':<22}{run(X_tr, X_te, 'raw'):>15.3f}")
sc = StandardScaler().fit(X_tr)                    # fit on TRAIN only
print(f"{'standardised':<22}{run(sc.transform(X_tr), sc.transform(X_te), 'std'):>15.3f}")
mm = MinMaxScaler().fit(X_tr)
print(f"{'min-max scaled':<22}{run(mm.transform(X_tr), mm.transform(X_te), 'mm'):>15.3f}")
print(f"\nmajority-class baseline: {max(y_te.mean(), 1 - y_te.mean()):.3f}")

Same model, same seed, same number of epochs. **One line different.**

The unscaled run does not merely score lower — it scores *below the majority-class
baseline*, which means it has learned something actively wrong rather than nothing.

## Part B — Why: the shape of the loss surface

This is not a mystery about neural networks; it is a fact about gradient descent on
badly conditioned problems. The **condition number** of the input covariance — the ratio of
its largest to smallest eigenvalue — says how elongated the loss valley is, and gradient
descent zig-zags across a narrow valley instead of running down it.

In [ ]:
def condition(A):
    ev = np.linalg.eigvalsh(np.cov(A, rowvar=False))
    return ev.max() / ev.min()

print(f"{'inputs':<22}{'condition number':>18}")
print(f"{'raw':<22}{condition(X_tr):>18,.0f}")
print(f"{'standardised':<22}{condition(sc.transform(X_tr)):>18,.1f}")
print(f"{'min-max scaled':<22}{condition(mm.transform(X_tr)):>18,.1f}")
print("\n1.0 would be a perfectly round bowl. The raw surface is a canyon.")

In [ ]:
# the same story as a learning curve: how many steps to reach a given loss
def logistic_gd(A, t, lr, steps=400):
    w, b = np.zeros(A.shape[1]), 0.0
    hist = []
    for _ in range(steps):
        p = 1 / (1 + np.exp(-np.clip(A @ w + b, -30, 30)))
        g = (p - t) / len(t)
        w -= lr * (A.T @ g)
        b -= lr * g.sum()
        hist.append(float(-(t * np.log(p + 1e-12) + (1 - t) * np.log(1 - p + 1e-12)).mean()))
    return np.array(hist)

print(f"{'inputs':<18}{'loss @10':>10}{'@100':>10}{'@400':>10}")
for label, A in (("raw", X_tr), ("standardised", sc.transform(X_tr))):
    h = logistic_gd(A, y_tr, lr=0.05)
    print(f"{label:<18}{h[9]:>10.4f}{h[99]:>10.4f}{h[399]:>10.4f}")
print("\nSame learning rate, same steps. The raw run does not merely fail to")
print("improve - it gets WORSE, because a step size that is sane for the age")
print("column is wildly too large for the salary column, and one shared learning")
print("rate has to serve both.")

## Part C — Which scaler, and the one rule that matters more than the choice

- **Standardisation** `(x - μ) / σ` → mean 0, std 1. Use when the maximum is unknown, or the
  data is roughly normal, or there are outliers a min/max would be dragged around by.
- **Min-max** `(x - min) / (max - min)` → exactly [0, 1]. Use when the bounds are *genuinely*
  known — GRE out of 340, an image pixel out of 255.
- **Images**: every pixel already shares one range, so just divide by 255. No scaler object
  is needed at all, because nothing has to be estimated.

The choice between them matters far less than the rule below.

In [ ]:
print("fit on TRAIN, transform BOTH - never fit on the test set\n")
print(f"{'':<26}{'mean':>9}{'std':>9}")
for label, A in (("train, standardised", sc.transform(X_tr)),
                 ("test,  standardised", sc.transform(X_te))):
    print(f"{label:<26}{A.mean():>9.3f}{A.std():>9.3f}")

print(f"\n{'':<26}{'min':>9}{'max':>9}")
for label, A in (("train, min-max", mm.transform(X_tr)), ("test,  min-max", mm.transform(X_te))):
    print(f"{label:<26}{A.min():>9.3f}{A.max():>9.3f}")
print("\nper-column test maxima under min-max:", np.round(mm.transform(X_te).max(0), 3))

The training set comes out with mean exactly 0 and std exactly 1, because that is what the
scaler was fitted to produce. **The test set does not** — mean 0.009, std 0.984. That is
correct behaviour, not a bug: the scaler learned its statistics from the training data
alone, so the test set has no reason to match them exactly.

Under min-max the test set happens to stay inside [0, 1] on this split, since the training
rows already covered the full age and salary ranges. It is *allowed* to escape, and on a
different split it would — a future applicant older or richer than anyone in training
legitimately scales above 1.

Fitting the scaler on all 400 rows would force every one of these numbers to be perfect —
and would leak information about the test set into training, making every score you report
slightly optimistic. **The imperfect numbers are the honest ones.**

## Part D — Where it stops mattering

Scaling is about the *input* layer. Once the data is inside the network, the same problem
can reappear between hidden layers as their activations drift — which is a different fix
(batch normalisation) applied at a different place.

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit on TRAIN only
X_test = scaler.transform(X_test)         # test is transformed, never fitted

# for images, where every pixel already shares one range:
X = X / 255.0
```

## Try it yourself

1. Scale *only* `EstimatedSalary` and leave `Age` raw. Does that fix it? What does the answer
   say about whether the problem is absolute magnitude or relative magnitude?
2. Lower the learning rate to 1e-6 for the raw data and raise `max_iter` to 20,000. Can you
   recover the scaled accuracy by brute force? How much compute did that cost?
3. Compute the condition number after scaling only one of the two columns.
4. Add an outlier — one row with a salary of 10,000,000 — and compare how `StandardScaler`
   and `MinMaxScaler` each react to it.